# FinTrust Digital Bank — Week 3 Model Development & Validation
**Takudzwa Jeffrey Gumbo | AnalystLab Africa Experience Lab | Data Science Track**

Purpose: refine the Week 2 baseline, test additional classifiers, analyse errors, interpret the candidate and leave one model ready for Week 4.

**Responsibility statement:** model outputs are decision-support evidence only. **Synthetic-data disclaimer:** `Risk_Review_Flag` is a synthetic educational review-prioritisation label and is not real fraud detection.

Method commitments: fixed 70/15/15 stratified split (seed 42); EDA/fitted statistics from training only; test untouched; preprocessing inside pipelines; primary ROC-AUC; recall ≥0.70; accuracy excluded from selection; replacement must exceed incumbent by more than its CV SD.

## 0. Setup and Google Drive data access
This notebook is self-contained: all Week 3 training functions and model-search logic are defined below. It does **not** import `week3_train.py` or `week3_analysis.py`.

In Google Colab, the notebook mounts Drive and reads the two raw FinTrust CSV files from `/content/drive/MyDrive/Analyst Lab Africa`. For reproducibility outside Colab, the same cell falls back to the repository's local `data/raw/` folder.

In [ ]:
from pathlib import Path
import json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, precision_recall_curve
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
SEED = 42

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Analyst Lab Africa')
    RAW = ROOT
    print('Google Drive mounted. Data root:', ROOT)
except ImportError:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    RAW = ROOT / 'data' / 'raw'
    print('Non-Colab runtime. Using local data root:', RAW)

OUT = ROOT / 'models' / 'week3'
FIGDIR = ROOT / 'reports' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

customer_path = RAW / 'FinTrust_Customer_Data.csv'
transaction_path = RAW / 'FinTrust_Transaction_Data.csv'
assert customer_path.exists(), f'Missing: {customer_path}'
assert transaction_path.exists(), f'Missing: {transaction_path}'
print('Input files found.')

## 1. Rebuild the fixed split before feature engineering
The original customer and transaction files are loaded and merged, then the fixed 70/15/15 stratified split is recreated with seed 42. The test partition is created here only to reserve it; it is not engineered, inspected, scored, or used in model selection.

In [ ]:
customers = pd.read_csv(customer_path)
transactions = pd.read_csv(transaction_path)
transactions['Transaction_DateTime'] = pd.to_datetime(
    transactions['Transaction_DateTime'], format='%m/%d/%Y %H:%M'
)

df = transactions.merge(customers, on='Customer_ID', validate='many_to_one')
df['target'] = (df['Risk_Review_Flag'] == 'Yes').astype(int)

train_val, test = train_test_split(
    df, test_size=0.15, stratify=df['target'], random_state=SEED
)
train, val = train_test_split(
    train_val, test_size=0.1765, stratify=train_val['target'], random_state=SEED
)

print('Train:', len(train), 'Validation:', len(val), 'Test reserved:', len(test))
print('Positive rates:', train.target.mean(), val.target.mean(), test.target.mean())

## 2. Training-only feature engineering and Week 3 refinement
Amount-band cut points are fitted from the training split only. The refined feature set removes redundant amount encodings, the confounded customer-relative amount ratio, weak day/weekend/count fields, and Age/Monthly_Income_Band. Gender remains excluded on fairness grounds.

In [ ]:
amount_bins = [-np.inf] + list(np.quantile(train['Amount_NGN'], [0.2, 0.4, 0.6, 0.8])) + [np.inf]

def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create Week 3 features using cut points already fitted on training data."""
    d = data.copy()
    d['Transaction_Hour'] = d['Transaction_DateTime'].dt.hour
    d['Is_Overnight'] = d['Transaction_Hour'].between(0, 5).astype(int)
    d['Amount_Band'] = pd.cut(
        d['Amount_NGN'], amount_bins, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
    ).astype(str)
    d['Is_International'] = (d['International_Transaction'] == 'Yes').astype(int)
    d['Tenure_Band'] = pd.cut(
        d['Tenure_Months'], [0, 12, 36, 60, 96],
        labels=['0-12m', '13-36m', '37-60m', '61-96m']
    ).astype(str)
    d['Engagement_Band'] = pd.cut(
        d['Digital_Engagement_Score'], [0, 40, 70, 100],
        labels=['Low', 'Medium', 'High']
    ).astype(str)
    return d

train = engineer_features(train)
val = engineer_features(val)

NUM = ['Transaction_Hour', 'Tenure_Months', 'Digital_Engagement_Score']
BIN = ['Is_Overnight', 'Is_International']
CAT = [
    'Transaction_Type', 'Channel', 'Device_Type', 'Location', 'Amount_Band',
    'Customer_Segment', 'Account_Type', 'Account_Status', 'Preferred_Channel',
    'Tenure_Band', 'Engagement_Band'
]
FEATURES = NUM + BIN + CAT

X, y = train[FEATURES], train['target']
Xv, yv = val[FEATURES], val['target']
print('Refined raw inputs:', FEATURES)
print('Count:', X.shape[1])

## 3. Pipeline and evaluation helpers
All fitted preprocessing stays inside `Pipeline`/`ColumnTransformer`, so cross-validation refits transformations independently in each fold.

In [ ]:
def make_pipeline(model) -> Pipeline:
    """Build a leakage-safe preprocessing and classification pipeline."""
    prep = ColumnTransformer([
        ('num', StandardScaler(), NUM),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), CAT),
    ], remainder='passthrough')
    return Pipeline([('prep', prep), ('model', model)])

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

def evaluate_model(name: str, pipeline: Pipeline):
    """Cross-validate on training data, fit training data, then evaluate validation at 0.5."""
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    pipeline.fit(X, y)
    prob = pipeline.predict_proba(Xv)[:, 1]
    pred = (prob >= 0.5).astype(int)
    result = {
        'model': name,
        'cv_auc_mean': float(scores.mean()),
        'cv_auc_std': float(scores.std()),
        'val_auc': float(roc_auc_score(yv, prob)),
        'val_pr_auc': float(average_precision_score(yv, prob)),
        'accuracy': float(accuracy_score(yv, pred)),
        'precision': float(precision_score(yv, pred, zero_division=0)),
        'recall': float(recall_score(yv, pred)),
        'f1': float(f1_score(yv, pred)),
    }
    return result, pipeline, prob

## 4. Refit the three Week 2 model families on the refined feature set
These provide the refined-feature reference point before the additional Week 3 searches.

In [ ]:
base_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight='balanced',
        random_state=SEED, n_jobs=-1
    ),
}
base_results = []
fitted = {}
for name, model in base_models.items():
    result, fitted_model, _ = evaluate_model(name, make_pipeline(model))
    base_results.append(result)
    fitted[name] = fitted_model
    joblib.dump(fitted_model, OUT / f"{name.replace(' ', '_')}.joblib")
    print(result)

with open(OUT / 'base.json', 'w') as f:
    json.dump(base_results, f, indent=2)

## 5. Tuned Random Forest
The tuned configuration identified during Week 3 development is refitted and cross-validated here directly, keeping the submission notebook self-contained and practical to rerun in Colab.


In [ ]:
# Refit the previously selected tuned-RF configuration inside this notebook.
rf_model = make_pipeline(RandomForestClassifier(
    n_estimators=300, min_samples_leaf=20, max_features='sqrt', max_depth=12,
    class_weight='balanced', random_state=SEED, n_jobs=-1
))
rf_result, rf_fitted, _ = evaluate_model('Tuned Random Forest', rf_model)
fitted['Tuned Random Forest'] = rf_fitted
joblib.dump(rf_fitted, OUT / 'Tuned_RF.joblib')
print(rf_result)


## 6. Additional model 1 — Gradient Boosting
The selected Gradient Boosting configuration is trained and evaluated directly in the notebook.


In [ ]:
# Gradient Boosting configuration selected by the Week 3 training search.
gb_model = make_pipeline(GradientBoostingClassifier(
    subsample=0.7, n_estimators=150, min_samples_leaf=10, max_depth=1,
    learning_rate=0.05, random_state=SEED
))
gb_result, gb_fitted, _ = evaluate_model('Gradient Boosting', gb_model)
fitted['Gradient Boosting'] = gb_fitted
joblib.dump(gb_fitted, OUT / 'GB.joblib')
print(gb_result)


## 7. Additional model 2 — Histogram Gradient Boosting
The selected `HistGradientBoostingClassifier` configuration is trained and evaluated directly using scikit-learn only.


In [ ]:
# Histogram Gradient Boosting configuration selected by the Week 3 training search.
hgb_model = make_pipeline(HistGradientBoostingClassifier(
    min_samples_leaf=20, max_leaf_nodes=15, max_iter=150, learning_rate=0.02,
    l2_regularization=1.0, random_state=SEED
))
hgb_result, hgb_fitted, _ = evaluate_model('HistGradientBoosting', hgb_model)
fitted['HistGradientBoosting'] = hgb_fitted
joblib.dump(hgb_fitted, OUT / 'HGB.joblib')
print(hgb_result)


## 8. Validation threshold tuning and model comparison
For each fitted model, choose the **highest validation threshold that still achieves recall ≥ 0.70**. Accuracy is reported descriptively but is not used for model selection.

In [ ]:
rows = []
probabilities = {}
for name, model in fitted.items():
    prob = model.predict_proba(Xv)[:, 1]
    probabilities[name] = prob
    precision_curve, recall_curve, thresholds = precision_recall_curve(yv, prob)
    eligible = np.where(recall_curve[:-1] >= 0.70)[0]
    threshold = float(thresholds[eligible[-1]])
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(yv, pred)
    rows.append({
        'Model': name, 'Threshold': threshold,
        'Accuracy': accuracy_score(yv, pred),
        'Precision': precision_score(yv, pred, zero_division=0),
        'Recall': recall_score(yv, pred), 'F1': f1_score(yv, pred),
        'ROC-AUC': roc_auc_score(yv, prob), 'PR-AUC': average_precision_score(yv, prob),
        'TN': int(cm[0, 0]), 'FP': int(cm[0, 1]), 'FN': int(cm[1, 0]), 'TP': int(cm[1, 1]),
    })
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / 'comparison_threshold_tuned.csv', index=False)
comparison

## 9. Candidate selection
Apply the pre-registered replacement rule literally: a challenger must improve on the incumbent by more than the incumbent's own cross-fold standard deviation. The test set remains untouched.

In [ ]:
incumbent_cv = next(r for r in base_results if r['model'] == 'Logistic Regression')
replacement_bar = incumbent_cv['cv_auc_mean'] + incumbent_cv['cv_auc_std']
print('Refined LR CV AUC:', incumbent_cv['cv_auc_mean'])
print('Refined LR CV SD:', incumbent_cv['cv_auc_std'])
print('Replacement bar:', replacement_bar)
print()
print('Validation comparison:')
print(comparison[['Model','Threshold','Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC']].to_string(index=False))

candidate_model = fitted['Logistic Regression']
candidate_prob = probabilities['Logistic Regression']
candidate_threshold = float(comparison.loc[comparison.Model == 'Logistic Regression', 'Threshold'].iloc[0])
candidate_pred = (candidate_prob >= candidate_threshold).astype(int)

## 10. Interpretation — permutation importance and logistic-regression coefficients
Permutation importance is retained for consistency with Week 2. The refined logistic-regression coefficients are also inspected to verify that removing redundant amount encodings resolved the earlier sign-instability problem.

In [ ]:
pi = permutation_importance(
    candidate_model, Xv, yv, n_repeats=15, random_state=SEED,
    scoring='roc_auc', n_jobs=2
)
importance = pd.DataFrame({
    'feature': Xv.columns,
    'importance': pi.importances_mean,
    'std': pi.importances_std,
}).sort_values('importance', ascending=False)
importance.to_csv(OUT / 'permutation_importance.csv', index=False)
print(importance.head(12).to_string(index=False))

feature_names = candidate_model.named_steps['prep'].get_feature_names_out()
coefficients = candidate_model.named_steps['model'].coef_[0]
coef = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
coef = coef.sort_values('coefficient', key=abs, ascending=False)
coef.to_csv(OUT / 'lr_coefficients.csv', index=False)
print()
print('Top absolute coefficients:')
print(coef.head(15).to_string(index=False))

## 11. Error analysis
Inspect false positives and false negatives on validation data, including performance by transaction type.

In [ ]:
errors = val[[
    'Transaction_ID', 'Transaction_Type', 'Amount_NGN', 'Channel',
    'International_Transaction', 'Transaction_Hour', 'Risk_Review_Flag'
]].copy()
errors['actual'] = yv.values
errors['probability'] = candidate_prob
errors['predicted'] = candidate_pred
errors['error'] = np.where(
    (errors.actual == 0) & (errors.predicted == 1), 'False Positive',
    np.where((errors.actual == 1) & (errors.predicted == 0), 'False Negative', 'Correct')
)
errors.to_csv(OUT / 'validation_scored_errors.csv', index=False)

segment_rows = []
for transaction_type, group in errors.groupby('Transaction_Type'):
    yt, yp = group.actual, group.predicted
    segment_rows.append({
        'Transaction_Type': transaction_type,
        'n': len(group), 'flagged': int(yt.sum()),
        'recall': recall_score(yt, yp, zero_division=0),
        'precision': precision_score(yt, yp, zero_division=0),
        'false_positives': int(((yt == 0) & (yp == 1)).sum()),
        'false_negatives': int(((yt == 1) & (yp == 0)).sum()),
    })
error_by_type = pd.DataFrame(segment_rows)
error_by_type.to_csv(OUT / 'error_by_transaction_type.csv', index=False)
error_by_type.sort_values('recall')

## 12. Week 3 figures

In [ ]:
rr = comparison.sort_values('ROC-AUC')
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(rr.Model, rr['ROC-AUC'])
ax.axvline(0.75, linestyle='--')
ax.set_xlabel('Validation ROC-AUC')
ax.set_title('Week 3 model comparison')
fig.tight_layout()
fig.savefig(FIGDIR / 'fig15_week3_model_comparison.png', dpi=200)
plt.show()

cm = confusion_matrix(yv, candidate_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center')
ax.set_xticks([0, 1], ['Not flagged', 'Flagged'])
ax.set_yticks([0, 1], ['Not flagged', 'Flagged'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Candidate LR confusion matrix (threshold={candidate_threshold:.4f})')
fig.tight_layout()
fig.savefig(FIGDIR / 'fig16_week3_candidate_confusion_matrix.png', dpi=200)
plt.show()

top = importance.head(10).sort_values('importance')
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.feature, top.importance, xerr=top['std'])
ax.set_xlabel('Mean drop in ROC-AUC')
ax.set_title('Candidate model permutation importance')
fig.tight_layout()
fig.savefig(FIGDIR / 'fig17_week3_permutation_importance.png', dpi=200)
plt.show()

## 13. Results summary

In [ ]:
summary = comparison[['Model','Threshold','Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC']].copy()
summary

## What is next — Week 4
Freeze the candidate pipeline and threshold, then score the untouched test set exactly once. Complete final error/fairness review, documentation and presentation.

**Synthetic-data / non-fraud disclaimer:** `Risk_Review_Flag` is a synthetic educational review-prioritisation label and is not real fraud detection.